In [15]:
from google.colab import files
uploaded = files.upload()

Saving kb.txt to kb (1).txt


In [3]:
!pip install spacy transformers sentence-transformers
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 58.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import spacy
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

In [16]:
with open("kb.txt", "r") as f:
    kb = f.read()

print(kb[:500])

{\rtf1\ansi\ansicpg1252\cocoartf2868
\cocoatextscaling0\cocoaplatform0{\fonttbl\f0\fswiss\fcharset0 Helvetica;}
{\colortbl;\red255\green255\blue255;}
{\*\expandedcolortbl;;}
\paperw11900\paperh16840\margl1440\margr1440\vieww11520\viewh8400\viewkind0
\pard\tx720\tx1440\tx2160\tx2880\tx3600\tx4320\tx5040\tx5760\tx6480\tx7200\tx7920\tx8640\pardirnatural\partightenfactor0

\f0\fs24 \cf0 Natural Language Processing (NLP) is a field of Artificial Intelligence that enables computers to understand and p


In [17]:
nlp = spacy.load("en_core_web_sm")

doc = nlp(kb)
sentences = [sent.text.strip() for sent in doc.sents]

print(sentences[:5])

['{\\rtf1\\ansi\\ansicpg1252\\cocoartf2868\n\\cocoatextscaling0\\cocoaplatform0{\\fonttbl\\f0\\fswiss\\fcharset0 Helvetica;}\n{\\colortbl;\\red255\\green255\\blue255;}\n{\\*\\expandedcolortbl;;}\n\\paperw11900\\paperh16840\\margl1440\\margr1440\\vieww11520\\viewh8400\\viewkind0\n\\pard\\tx720\\tx1440\\tx2160\\tx2880\\tx3600\\tx4320\\tx5040\\tx5760\\tx6480\\tx7200\\tx7920\\tx8640\\pardirnatural\\partightenfactor0\n\n\\f0\\fs24 \\cf0 Natural Language Processing (NLP) is a field of Artificial Intelligence that enables computers to understand and process human language.\\\nIt involves tasks such as tokenization, part-of-speech tagging, named entity recognition, and parsing.\\\n\\\nspaCy is an open-source Python library used for advanced NLP tasks.', 'It provides efficient tools for text processing and linguistic analysis.\\\n\\\nTransformers are deep learning models that use attention mechanisms to understand context in text.', 'They are widely used in applications such as question answeri

In [18]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
qa_model = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [20]:
def get_answer(question):
    # Convert question to embedding
    q_embed = embed_model.encode(question)

    # Convert sentences to embeddings
    s_embed = embed_model.encode(sentences)

    # Compute similarity
    scores = util.cos_sim(q_embed, s_embed)

    # Get best sentence
    best_idx = scores.argmax()
    context = sentences[best_idx]

    # Get answer from QA model
    result = qa_model(question=question, context=context)

    return result["answer"], context

In [21]:
question = "What is NLP?"

answer, context = get_answer(question)

print("Question:", question)
print("Context:", context)
print("Answer:", answer)

Question: What is NLP?
Context: {\rtf1\ansi\ansicpg1252\cocoartf2868
\cocoatextscaling0\cocoaplatform0{\fonttbl\f0\fswiss\fcharset0 Helvetica;}
{\colortbl;\red255\green255\blue255;}
{\*\expandedcolortbl;;}
\paperw11900\paperh16840\margl1440\margr1440\vieww11520\viewh8400\viewkind0
\pard\tx720\tx1440\tx2160\tx2880\tx3600\tx4320\tx5040\tx5760\tx6480\tx7200\tx7920\tx8640\pardirnatural\partightenfactor0

\f0\fs24 \cf0 Natural Language Processing (NLP) is a field of Artificial Intelligence that enables computers to understand and process human language.\
It involves tasks such as tokenization, part-of-speech tagging, named entity recognition, and parsing.\
\
spaCy is an open-source Python library used for advanced NLP tasks.
Answer: Natural Language Processing


In [23]:
while True:
    q = input("Ask a question (type 'exit' to stop): ")
    if q.lower() == "exit":
        break

    ans, ctx = get_answer(q)

    print("Answer:", ans)
    print("Context used:", ctx)
    print("-" * 50)

Ask a question (type 'exit' to stop): What is Machine Learning?
Answer: a subset of Artificial Intelligence
Context used: They are widely used in applications such as question answering, translation, and text generation.\
\
Machine Learning is a subset of Artificial Intelligence that allows systems to learn from data and improve performance without explicit programming.\
\
Deep Learning is a specialized branch of Machine Learning that uses neural networks with multiple layers.\
\
Question Answering systems are designed to automatically respond to user queries by extracting relevant information from a knowledge base.}
--------------------------------------------------
Ask a question (type 'exit' to stop): What is Deep Learning?
Answer: a specialized branch of Machine Learning
Context used: They are widely used in applications such as question answering, translation, and text generation.\
\
Machine Learning is a subset of Artificial Intelligence that allows systems to learn from data and